# 49 — Region-Constrained Design Search

**Subtitle:** Topology Specifies Robust Engineering

**Engineering question**

How should an engineer search a design space when logical computation depends on preserving the largest connected admissible region?

Notebook 43 established the reusable specification:

\[
\boxed{
\text{Design}
\rightarrow
\Omega
\rightarrow
\Omega_c
\rightarrow
d(\partial\Omega_c)
\rightarrow
x^\*
\rightarrow
\text{Logical Computation}
}
\]

Notebook 49 turns that specification into an explicit search procedure.

## Setup

In [ ]:
from pathlib import Path
import json, zipfile
from collections import deque
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
FIG = ROOT / "figures"
RES = ROOT / "results"
CSV = RES / "csv"
JS = RES / "json"
for p in [FIG, RES, CSV, JS]:
    p.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12

def savefig(fig, filename, dpi=220):
    path = FIG / filename
    fig.tight_layout()
    fig.savefig(path, dpi=dpi)
    print(f"saved: {path}")
    return path

def save_table(df, stem):
    df.to_csv(CSV / f"{stem}.csv", index=False)
    df.to_json(JS / f"{stem}.json", orient="records", indent=2)

def grid(n=220):
    x = np.linspace(0,1,n)
    y = np.linspace(0,1,n)
    X,Y = np.meshgrid(x,y)
    return x,y,X,Y

def connected_components(mask):
    visited = np.zeros_like(mask, dtype=bool)
    h,w = mask.shape
    comps=[]
    for i in range(h):
        for j in range(w):
            if mask[i,j] and not visited[i,j]:
                q=deque([(i,j)])
                visited[i,j]=True
                pts=[]
                while q:
                    a,b=q.popleft()
                    pts.append((a,b))
                    for da,db in [(1,0),(-1,0),(0,1),(0,-1)]:
                        na,nb=a+da,b+db
                        if 0<=na<h and 0<=nb<w and mask[na,nb] and not visited[na,nb]:
                            visited[na,nb]=True
                            q.append((na,nb))
                comps.append(pts)
    comps.sort(key=len, reverse=True)
    return comps

def boundary_distance(mask):
    h,w=mask.shape
    INF=h+w+10
    dist=np.full((h,w), INF, dtype=float)
    q=deque()
    for i in range(h):
        for j in range(w):
            if not mask[i,j]:
                dist[i,j]=0
                q.append((i,j))
    while q:
        i,j=q.popleft()
        for di,dj in [(1,0),(-1,0),(0,1),(0,-1)]:
            ni,nj=i+di,j+dj
            if 0<=ni<h and 0<=nj<w and dist[ni,nj] > dist[i,j]+1:
                dist[ni,nj]=dist[i,j]+1
                q.append((ni,nj))
    return dist

def admissibility_score(X,Y,design):
    drive=np.clip(X+design.get("drive_shift",0),0,1.2)
    loss=np.clip(Y*design.get("loss_scale",1),0,1.4)
    enough=1/(1+np.exp(-18*(drive-0.30)))
    loss_survive=np.exp(-2.8*loss)
    over=np.exp(-9*np.maximum(drive-0.86,0)**2)
    timing=np.exp(-design.get("timing_scale",1)*5*np.maximum(drive+0.74*loss-1.08,0)**2)
    cal=np.exp(-4*design.get("calibration",0)**2)
    broad=np.exp(-((drive-0.62)**2/0.27+(loss-0.20)**2/0.18))
    defects=1-design.get("routing_noise",0)*np.sin(10*X)*np.sin(8*Y)
    Z=design.get("coupling",1)*design.get("detection",1)*enough*loss_survive*over*timing*cal*(0.5+0.5*broad)*defects
    Z=np.clip(Z,0,None)
    return Z/(Z.max()+1e-12)

def region_metrics(design,n=200,threshold=0.50):
    x,y,X,Y=grid(n)
    Z=admissibility_score(X,Y,design)
    mask=Z>=threshold
    comps=connected_components(mask)
    dist=boundary_distance(mask)
    if comps:
        pts=np.array(comps[0])
        largest=len(comps[0])/mask.size
        avg=dist[pts[:,0],pts[:,1]].mean()/max(mask.shape)
        mx=dist[pts[:,0],pts[:,1]].max()/max(mask.shape)
        best=pts[np.argmax(dist[pts[:,0],pts[:,1]])]
        bx=x[best[1]]; by=y[best[0]]
    else:
        largest=avg=mx=bx=by=0
    return dict(x=x,y=y,X=X,Y=Y,Z=Z,mask=mask,components=comps,dist=dist,
                admissible_area=mask.mean(),largest_component=largest,component_count=len(comps),
                avg_margin=avg,max_margin=mx,best_x=bx,best_y=by)

## 1. Region-constrained specification

\[
\boxed{
\text{Design}
\rightarrow
\Omega
\rightarrow
\Omega_c
\rightarrow
d(\partial\Omega_c)
\rightarrow
x^\*
}
\]

In [ ]:
fig, ax = plt.subplots(figsize=(13.5,4.8))
ax.axis("off")
items=[("Design","candidate hardware"),("Ω","admissible region"),("Ωc","largest connected component"),("d(∂Ωc)","distance to boundary"),("x*","maximum-margin point"),("Logical\nComputation","execution")]
xs=np.linspace(.08,.92,len(items)); y=.56; w=.135; h=.28
for i,(title,sub) in enumerate(items):
    ax.add_patch(Rectangle((xs[i]-w/2,y-h/2),w,h,fill=False,linewidth=2.3))
    ax.text(xs[i],y+.045,title,ha="center",va="center",fontsize=12,weight="bold")
    ax.text(xs[i],y-.075,sub,ha="center",va="center",fontsize=8.7)
    if i<len(items)-1:
        ax.annotate("",xy=(xs[i+1]-w/2-.008,y),xytext=(xs[i]+w/2+.008,y),arrowprops=dict(arrowstyle="->",linewidth=2.2))
ax.text(.5,.17,"Preserve the region before selecting the point.",ha="center",fontsize=12)
ax.set_title("Region-Constrained Design Search Specification",fontsize=18)
savefig(fig,"49_01_design_specification.png")
plt.show()

## 2. Point search versus region search

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(13,4.8))
for ax in axes: ax.axis("off")
left=[("candidate",.82),("evaluate score",.62),("move uphill",.42),("repeat",.22)]
right=[("candidate design",.86),("admissible region Ω",.68),("largest component Ωc",.50),("distance transform d(∂Ωc)",.32),("choose x*",.14)]
for label,yy in left:
    axes[0].add_patch(Rectangle((.28,yy-.06),.44,.12,fill=False,linewidth=2)); axes[0].text(.5,yy,label,ha="center",va="center",fontsize=11)
for i in range(len(left)-1):
    axes[0].annotate("",xy=(.5,left[i+1][1]+.07),xytext=(.5,left[i][1]-.07),arrowprops=dict(arrowstyle="->",linewidth=2))
axes[0].set_title("Point optimization",fontsize=15); axes[0].text(.5,.02,"optimizes a score at one point",ha="center",fontsize=10)
for label,yy in right:
    axes[1].add_patch(Rectangle((.22,yy-.055),.56,.11,fill=False,linewidth=2)); axes[1].text(.5,yy,label,ha="center",va="center",fontsize=10.5)
for i in range(len(right)-1):
    axes[1].annotate("",xy=(.5,right[i+1][1]+.065),xytext=(.5,right[i][1]-.065),arrowprops=dict(arrowstyle="->",linewidth=2))
axes[1].set_title("Region-constrained search",fontsize=15); axes[1].text(.5,.02,"optimizes topology before choosing a point",ha="center",fontsize=10)
fig.suptitle("Point Search vs Region Search",fontsize=18)
savefig(fig,"49_02_point_vs_region_search.png")
plt.show()

## 3. Region objective

\[
J(D)=\alpha |\Omega_c(D)|+\beta M(D)-\gamma C(D)-\delta K(D)
\]

In [ ]:
fig,ax=plt.subplots(figsize=(12,5)); ax.axis("off")
ax.text(.5,.70,r"$J(D)=\alpha |\Omega_c(D)| + \beta M(D) - \gamma C(D) - \delta K(D)$",ha="center",fontsize=23)
terms=[("|Ωc|","largest connected\nadmissible area",.18),("M","maximum robustness\nmargin",.38),("C","hardware\ncost",.60),("K","control\ncomplexity",.80)]
for sym,meaning,x0 in terms:
    ax.add_patch(Rectangle((x0-.085,.24),.17,.22,fill=False,linewidth=2))
    ax.text(x0,.39,sym,ha="center",va="center",fontsize=18,weight="bold")
    ax.text(x0,.29,meaning,ha="center",va="center",fontsize=10)
ax.text(.5,.08,"The objective optimizes the region, then selects the operating point.",ha="center",fontsize=12)
ax.set_title("Region-Constrained Objective Function",fontsize=18)
savefig(fig,"49_03_region_objective.png")
plt.show()

## 4. Candidate design metrics

In [ ]:
candidate_designs=[
{"name":"single cavity","drive_shift":-.06,"loss_scale":1.28,"timing_scale":1.20,"calibration":.08,"routing_noise":.06,"detection":.92,"coupling":.94,"cost":2,"control_complexity":2},
{"name":"coupled resonators","drive_shift":-.02,"loss_scale":1.10,"timing_scale":1.05,"calibration":.05,"routing_noise":.04,"detection":.95,"coupling":1.00,"cost":4,"control_complexity":4},
{"name":"programmable mesh","drive_shift":.02,"loss_scale":1.00,"timing_scale":.90,"calibration":.04,"routing_noise":.02,"detection":.98,"coupling":1.03,"cost":7,"control_complexity":7},
{"name":"hybrid chip","drive_shift":.04,"loss_scale":.82,"timing_scale":.72,"calibration":.02,"routing_noise":.01,"detection":1.02,"coupling":1.08,"cost":9,"control_complexity":8},
]
alpha,beta,gamma,delta=1.0,1.4,.018,.014
rows=[]
for d in candidate_designs:
    m=region_metrics(d,n=210)
    score=alpha*m["largest_component"]+beta*m["max_margin"]-gamma*d["cost"]-delta*d["control_complexity"]
    rows.append({"design":d["name"],"admissible_area":m["admissible_area"],"largest_component":m["largest_component"],"component_count":m["component_count"],"max_margin":m["max_margin"],"avg_margin":m["avg_margin"],"cost":d["cost"],"control_complexity":d["control_complexity"],"score":score,"x_star":m["best_x"],"y_star":m["best_y"]})
candidate_metrics=pd.DataFrame(rows).sort_values("score",ascending=False)
save_table(candidate_metrics,"49_candidate_design_metrics")
candidate_metrics

In [ ]:
fig,ax=plt.subplots(figsize=(14,4.6)); ax.axis("off")
show_cols=["design","largest_component","max_margin","cost","control_complexity","score","x_star","y_star"]
display_df=candidate_metrics[show_cols].copy()
for c in ["largest_component","max_margin","score","x_star","y_star"]:
    display_df[c]=display_df[c].map(lambda v:f"{v:.3f}")
table=ax.table(cellText=display_df.values,colLabels=display_df.columns,loc="center",cellLoc="center")
table.auto_set_font_size(False); table.set_fontsize(9.8); table.scale(1.05,1.65)
for (r,c),cell in table.get_celld().items():
    cell.set_linewidth(1.1)
    if r==0: cell.set_text_props(weight="bold")
    if c==0 and r>0: cell.set_text_props(ha="left")
ax.set_title("Architecture Comparison from Region Metrics",fontsize=17,pad=18)
savefig(fig,"49_04_architecture_region_metrics.png")
plt.show()

## 5. Design space evolution

In [ ]:
iteration_designs=[
{"name":"initial","drive_shift":-.09,"loss_scale":1.35,"timing_scale":1.25,"calibration":.08,"routing_noise":.08,"detection":.92,"coupling":.92,"cost":2,"control_complexity":2},
{"name":"admitted","drive_shift":-.04,"loss_scale":1.20,"timing_scale":1.12,"calibration":.06,"routing_noise":.05,"detection":.95,"coupling":.98,"cost":3,"control_complexity":3},
{"name":"connected","drive_shift":0.00,"loss_scale":1.02,"timing_scale":.95,"calibration":.04,"routing_noise":.025,"detection":.99,"coupling":1.02,"cost":5,"control_complexity":5},
{"name":"expanded","drive_shift":.04,"loss_scale":.84,"timing_scale":.76,"calibration":.02,"routing_noise":.01,"detection":1.02,"coupling":1.08,"cost":8,"control_complexity":7},
]
fig,axes=plt.subplots(1,4,figsize=(16,4.2),sharex=True,sharey=True)
evo=[]
for ax,d in zip(axes,iteration_designs):
    m=region_metrics(d,n=180)
    ax.imshow(m["Z"],origin="lower",extent=[0,1,0,1],vmin=0,vmax=1,aspect="auto")
    ax.contour(m["X"],m["Y"],m["Z"],levels=[.50],colors="black",linewidths=2.1)
    ax.scatter([m["best_x"]],[m["best_y"]],s=110,color="red",zorder=4)
    ax.set_title(d["name"]); ax.set_xlabel("drive")
    if ax is axes[0]: ax.set_ylabel("loss")
    ax.text(.05,.88,f"A={m['largest_component']:.2f}",transform=ax.transAxes,fontsize=10)
    ax.text(.05,.78,f"M={m['max_margin']:.2f}",transform=ax.transAxes,fontsize=10)
    evo.append({"iteration":d["name"],"largest_component":m["largest_component"],"max_margin":m["max_margin"],"component_count":m["component_count"]})
fig.suptitle("Design Space Evolution: Expand the Region Before Selecting the Point",fontsize=17,y=1.03)
savefig(fig,"49_05_design_space_evolution.png")
plt.show()
evolution=pd.DataFrame(evo)
save_table(evolution,"49_design_evolution")
evolution

## 6. Distance transform selects \(x^*\)

In [ ]:
best_design_name=candidate_metrics.iloc[0]["design"]
best_design=next(d for d in candidate_designs if d["name"]==best_design_name)
m=region_metrics(best_design,n=240)
dist_norm=m["dist"]/(m["dist"].max()+1e-12)
dist_masked=np.where(m["mask"],dist_norm,np.nan)
fig,ax=plt.subplots(figsize=(10,6.2))
im=ax.imshow(dist_masked,origin="lower",extent=[0,1,0,1],aspect="auto",vmin=0,vmax=1)
ax.contour(m["X"],m["Y"],m["Z"],levels=[.50],colors="black",linewidths=2.2)
ax.scatter([m["best_x"]],[m["best_y"]],s=240,color="red",zorder=4)
ax.annotate("x* maximum-margin point",xy=(m["best_x"],m["best_y"]),xytext=(m["best_x"]+.10,m["best_y"]+.19),arrowprops=dict(arrowstyle="->",linewidth=2),fontsize=11)
ax.set_title("Distance Transform Selects the Maximum-Margin Operating Point",fontsize=17)
ax.set_xlabel("drive"); ax.set_ylabel("loss")
cbar=fig.colorbar(im,ax=ax); cbar.set_label("normalized distance to failure boundary")
savefig(fig,"49_06_distance_transform.png")
plt.show()

## 7. Search algorithm and pseudocode

In [ ]:
fig,ax=plt.subplots(figsize=(8.8,7.2)); ax.axis("off")
steps=["Generate design","Physics simulation","Admissibility mask Ω","Connected components Ωc","Distance transform d(∂Ωc)","Region score J(D)","Keep / reject design"]
ys=np.linspace(.88,.12,len(steps))
for i,(step,yy) in enumerate(zip(steps,ys)):
    ax.add_patch(Rectangle((.24,yy-.045),.52,.09,fill=False,linewidth=2))
    ax.text(.50,yy,step,ha="center",va="center",fontsize=11)
    if i<len(steps)-1:
        ax.annotate("",xy=(.5,ys[i+1]+.052),xytext=(.5,yy-.052),arrowprops=dict(arrowstyle="->",linewidth=2))
ax.set_title("Region-Constrained Search Algorithm",fontsize=17)
savefig(fig,"49_07_search_algorithm.png")
plt.show()

In [ ]:
pseudocode=["best = None","for design in candidate_designs:","    Ω = admissible_region(design)","    Ωc = largest_connected_component(Ω)","    M = max_distance_to_boundary(Ωc)","    J = α*area(Ωc) + β*M - γ*cost(design) - δ*complexity(design)","    best = update(best, design, J)","x* = maximum_margin_point(best.Ωc)","return best.design, x*"]
fig,ax=plt.subplots(figsize=(11,5.2)); ax.axis("off")
ax.add_patch(Rectangle((.08,.08),.84,.82,fill=False,linewidth=2))
for i,line in enumerate(pseudocode): ax.text(.13,.82-i*.09,line,family="monospace",fontsize=12)
ax.set_title("Pseudocode: Search Over Regions, Not Points",fontsize=17)
savefig(fig,"49_08_pseudocode.png")
plt.show()

## 8. Search evolution metrics

In [ ]:
iterations=np.arange(1,len(evolution)+1)
fig,ax=plt.subplots(figsize=(8.8,5.4))
ax.plot(iterations,evolution["largest_component"],marker="o",linewidth=2.5,label="largest connected region")
ax.plot(iterations,evolution["max_margin"],marker="s",linewidth=2.5,label="maximum margin")
ax.set_xticks(iterations); ax.set_xticklabels(evolution["iteration"])
ax.set_ylim(0,max(evolution["largest_component"].max(),evolution["max_margin"].max())*1.25)
ax.set_title("Search Progress Is Measured by Topology",fontsize=17)
ax.set_ylabel("relative metric"); ax.grid(alpha=.25); ax.legend()
savefig(fig,"49_09_search_evolution_metrics.png")
plt.show()

## 9. Engineering tradeoffs

In [ ]:
trade=candidate_metrics.copy()
trade["normalized_region"]=trade["largest_component"]/trade["largest_component"].max()
trade["normalized_margin"]=trade["max_margin"]/trade["max_margin"].max()
trade["cost_penalty"]=trade["cost"]/trade["cost"].max()
trade["complexity_penalty"]=trade["control_complexity"]/trade["control_complexity"].max()
metrics=["normalized_region","normalized_margin","cost_penalty","complexity_penalty"]
angles=np.linspace(0,2*np.pi,len(metrics),endpoint=False).tolist(); angles+=angles[:1]
fig=plt.figure(figsize=(8,8)); ax=fig.add_subplot(111,polar=True)
for _,row in trade.iterrows():
    vals=[row[m] for m in metrics]; vals+=vals[:1]
    ax.plot(angles,vals,linewidth=2,label=row["design"])
ax.set_xticks(angles[:-1]); ax.set_xticklabels(["region","margin","cost","complexity"]); ax.set_yticklabels([])
ax.set_title("Engineering Tradeoffs in Region-Constrained Search",fontsize=16,pad=20)
ax.legend(loc="upper right",bbox_to_anchor=(1.35,1.10))
savefig(fig,"49_10_engineering_tradeoffs.png")
plt.show()

## 10. Region search pipeline and specification

In [ ]:
fig,ax=plt.subplots(figsize=(14,4.8)); ax.axis("off")
pipeline=[("Physics","what can exist?"),("Resources","what was generated?"),("Admissibility","what survives?"),("Topology","largest connected region"),("Distance\nGeometry","margin to boundary"),("Region\nScore","J(D)"),("Operating\nPoint","x*"),("Logical\nComputation","execution")]
xs=np.linspace(.06,.94,len(pipeline)); y=.55; w=.105; h=.28
for i,(title,sub) in enumerate(pipeline):
    ax.add_patch(Rectangle((xs[i]-w/2,y-h/2),w,h,fill=False,linewidth=2))
    ax.text(xs[i],y+.045,title,ha="center",va="center",fontsize=9.5,weight="bold")
    ax.text(xs[i],y-.070,sub,ha="center",va="center",fontsize=7.8)
    if i<len(pipeline)-1:
        ax.annotate("",xy=(xs[i+1]-w/2-.007,y),xytext=(xs[i]+w/2+.007,y),arrowprops=dict(arrowstyle="->",linewidth=2))
ax.text(.5,.17,"Topology specifies robustness. Robustness specifies operation. Operation specifies computation.",ha="center",fontsize=12)
ax.set_title("Region Search Pipeline",fontsize=18)
savefig(fig,"49_11_region_search_pipeline.png")
plt.show()

In [ ]:
spec=pd.DataFrame([
{"Symbol":"Design","Meaning":"candidate architecture or control policy","Computation":"input to search"},
{"Symbol":"Ω","Meaning":"admissible region","Computation":"thresholded specification mask"},
{"Symbol":"Ωc","Meaning":"largest connected admissible component","Computation":"connected-component analysis"},
{"Symbol":"d(∂Ωc)","Meaning":"distance to failure boundary","Computation":"distance transform"},
{"Symbol":"x*","Meaning":"maximum-margin operating point","Computation":"argmax distance inside Ωc"},
{"Symbol":"Logical Computation","Meaning":"fault-tolerant execution","Computation":"selected from robust region"},
])
save_table(spec,"49_engineering_specification")
fig,ax=plt.subplots(figsize=(14,4.4)); ax.axis("off")
table=ax.table(cellText=spec.values,colLabels=spec.columns,loc="center",cellLoc="center")
table.auto_set_font_size(False); table.set_fontsize(9.4); table.scale(1.05,1.65)
for (r,c),cell in table.get_celld().items():
    cell.set_linewidth(1.1)
    if r==0: cell.set_text_props(weight="bold")
ax.set_title("Engineering Specification for Region-Constrained Design Search",fontsize=17,pad=18)
savefig(fig,"49_12_engineering_specification.png")
plt.show()
spec

## Export bundle

In [ ]:
review={"candidate_metrics":candidate_metrics.to_dict(orient="records"),"design_evolution":evolution.to_dict(orient="records"),"engineering_specification":spec.to_dict(orient="records")}
with open(JS/"49_review_bundle.json","w",encoding="utf-8") as f: json.dump(review,f,indent=2)
zip_path=RES/"49_outputs.zip"
files_to_zip=list(FIG.glob("49_*.png"))+list(CSV.glob("49_*.csv"))+list(JS.glob("49_*.json"))
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for file in files_to_zip:
        if file.exists(): z.write(file,file.relative_to(ROOT))
print(f"Created: {zip_path}")
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    from IPython.display import FileLink, display
    display(FileLink(str(zip_path)))

## Takeaways

- Region-constrained design search optimizes topology before selecting an operating point.
- The leading object is \( \Omega_c \), the largest connected admissible component.
- The maximum-margin operating point \(x^\*\) is selected after distance geometry is evaluated.
- Cost and control complexity belong in the objective, but they should not replace admissibility.

\[
\boxed{
\text{Design}
\rightarrow
\Omega
\rightarrow
\Omega_c
\rightarrow
d(\partial\Omega_c)
\rightarrow
x^\*
\rightarrow
\text{Logical Computation}
}
\]